# Text classification - spam detector

We start by loading all necessary packages and helper functions

In [1]:
# Load packages and helper functions
%run data_prep_tools.py
%run EDA_tools.py


# Data

We will use the [SpamAssassin](https://spamassassin.apache.org/) public email corpus. This dataset contains ~6'000 labeled emails. If you want to learn more about this dataset, check [this](https://spamassassin.apache.org/old/publiccorpus/). (*Note: Datasets of text are called corpora and samples are called documents.*) 

The dataset has been downloaded for you and is available in the *data* folder.

The dataset has been labelled, i.e. we are told whether an email has been designated as spam, .e.g. if it was flagged by a user, or whether it is considered an example of regular emails (non-spam, also called "ham"). 

Our goal is to explore and compare various features space and machine learning approaches. The use of spam emails is just for demonstration and learning purpose as it is a text-based example that everyone is easily familiar with and that allows us to highlight different stages of developing a machine learning application and the decision making processes involved along the way.


## Data preparation - Overview

In this notebook we will explore the dataset, do a first analysis and prepare it for different machine learning tasks.

### Task 

We will process the raw data, clean the text and extract additional features ain order to prepare it for further analysis and for building our machine learning models.

### Notebook overview

* Load the data
* Text preprocessing
* Feature extraction
* Store cleaned data


## Load the data

In [2]:
# Load the data
df_source = load_source_data()

8546 emails loaded
Cleaning data set:
2710 duplicate emails found and removed
4 empty emails found and removed

5832 emails remaining

Number of columns: 2
Columns names:
spam_label, text


<div class="alert alert-success">
<h3>Questions</h3>

Look at the example in the next cell below and try to answer the questions below
    
__Q1.__ What parts of the text do you think are noise?
   
__Q2.__ What should we do with these parts of the text?
</div>


In [3]:
# If you rerun this cell multiple times you get different samples displayed each time
# OR you can replace the number 3 with a number of your choice
display(df_source.sample(3))

,spam_label,text
4969,0,">I currently have spam being filtered for about 50 users via spamd. >Works perfectly! However, is there anyone out there providing reports >to their users about what was filetered? Maybe just something like >date/time, From, and subject? Yup! You can check: http://www.cs.ait.ac.th/laboratory/email/quarantine.shtml Though it uses procmail! Olivier ------------------------------------------------------- This sf.net email is sponsored by:ThinkGeek Welcome to geek heaven. http://thinkgeek.com/sf _______________________________________________ Spamassassin-talk mailing list Spamassassin-talk@lists.sourceforge.net https://lists.sourceforge.net/lists/listinfo/spamassassin-talk"
5101,0,"use Perl Daily Headline Mailer Damian Conway to speak in London posted by ziggy on Thursday August 08, @15:21 (damian) http://use.perl.org/article.pl?sid=02/08/08/1923211 Meeting in Budapest posted by ziggy on Thursday August 08, @17:24 (groups) http://use.perl.org/article.pl?sid=02/08/08/1927253 Copyright 1997-2002 pudge. All rights reserved. ====================================================================== You have received this message because you subscribed to it on use Perl. To stop receiving this and other messages from use Perl, or to add more messages or change your preferences, please go to your user page. http://use.perl.org/my/messages/ You can log in and change your preferences from there."
539,0,"<HTML><HEAD><TITLE>Buyer's Alert | July 18, 2002</TITLE> </HEAD> <body style=""margin:8px 9px 9px 12px"" bgcolor=""#ffffff"" background=""http://techupdate.zdnet.com/techupdate/i/bg_232850.gif"" link=""#003399"" alink=""#cc0000"" vlink=""#666699""> <CENTER><A name=top></A> <TABLE cellSpacing=0 cellPadding=0 width=612 bgColor=#232850 border=0> <TBODY> <TR vAlign=bottom> <TD width=440 colSpan=4><IMG height=60 alt=""ZDNet Buyers Alert Newsletter"" src=""http://www.cnet.com/i/sh/nl/hd_buyers_alert.gif"" width=440></TD> <TD vAlign=top align=middle width=160 bgColor=#ffffff rowSpan=2><!-- ad --> <div align=""center""><img src=""http://i.i.com.com/cnwk.1d/Ads/common/advertisement.gif"" width=""120"" height=""10"" border=""0""></br> <a href=""http://clickthru.online.com/Click?q=fc-WpHmQVwTauRU4pZa9-0t3VVat9yR"" target=""new""> <img src=""http://www.techrepublic.com/images/ads/Toshiba.sky.2400freebag_160x600.gif"" width=""160"" height=""600"" border=""0""> </a></div> <!-- /ad --><!-- rebates --> <br><img src=""http://www.cnet.com/i/nl/zd-ba/rebates.gif"" width=160 height=18 alt=""Rebates""><br> <table width=160 bgcolor=""#eeeeee"" cellpadding=5 cellspacing=0 border=0> <tr><td><font face=""arial, helvetica"" size=""-2"">When ordering, make sure the reseller/manufacturer provides the rebate coupon you need. <br><br> <br><br> <font face=""arial, helvetica"" size=""-2"">Expires July 31, 2002<br> <li><a href=""http://clickthru.online.com/Click?q=11-l8riI2-aPLZhppqX3VynhNBuCqlR"" >Get a car cassette adapter and power adapter FREE with Rio Volt SP250</a> <br><br> <font face=""arial, helvetica"" size=""-2"">Expires August 31, 2002<br> <li><a href=""http://clickthru.online.com/Click?q=27-z9ZmIqI14RundZFSkMRUVolYrBeR"" >$20 off SideWinder Strategic Commander</a> <br><br> <font face=""arial, helvetica"" size=""-2"">Expires September 30, 2002<br> <li><a href=""http://clickthru.online.com/Click?q=3c-tK4RIcIw_GwZ-o1Y0VXXjhrHNHyR"" >$20 off Rio 600 64MB</a> <li><a href=""http://clickthru.online.com/Click?q=51-QgUWIjJWhgT2xbBJlh9GpMUHM3FR"" >$20 off Nik..."


## 💡 Observations

1. There are some items in the text that should be removed or transformed to make it readable and interpretable.  
   Here are some suggestions:
* HTML tags 
* URLs
* E-mail addresses
* Punctuation marks, digits (e.g. 2002, 1.1, ...)
* Multiple whitespaces
* Case conversion (e.g. Dog vs dog, ...)
* English STOPWORDS (e.g. a, is, my, i, all, and, by...)
* ...

2. From experience, we know that the number of occurrences of the above items (HTML tags, URLs, etc) can be helpful to distinguish spam from non-spam. Similarly, the length of the emails and the frequency of punctuation marks or upper case letters could also give us clues as to whether we are dealing with spam or not.

## Text cleaning and preprocessing

Good text preprocessing is an essential part of every NLP project. It is the first step in the machine learning pipeline and it is important to get it right. The goal of text preprocessing is to transform the raw text into a format that can be used by machine learning algorithms.

Our overall goal is to build models that can help us distinguish non-spam from spam. 

The examples above have shown us that some samples are quite messy and contain a lot of content unnecessary for understanding the text as a human, i.e. they contain "noise". As a first step we will **clean** and **standardize** raw text. Our aim is to keep as many **informative** words as possible, while discarding the **uniformative** ones. Removing the noise from our texts will help to improve the accuracy of our models.

We thus need to identify which parts of the text are acting as "**noise**" in our text and remove it.

The *clean_corpus* function below will take care of the parts raised in the 1st set of observations.

In [4]:
df_cleaned = clean_corpus(df_source)

Number of samples: 5832
Number of columns: 3
Columns names:
spam_label, text, text_cleaned

Number of duplicate cleaned texts found: 280
Number of empty texts found: 28

Email texts cleaned
Number of samples: 5832


In [5]:
# Let's look at some examples.
# You can rerun this cell to get a different sample
show_clean_text(df_cleaned)


Original document:

> Date: Mon, 30 Sep 2002 14:00:12 -0400 (EDT) > From: Dayv Gastonguay <noghri@nauticom.net> > > I
just installed razor 2.152 on a FreeBSD 4.4-RELEASE box and having > problems with razor-check. Any
time razor-check is run, (with or without > arguments), i get this error: > > Can't use an undefined
value as a symbol reference at >
/usr/local/lib/perl5/site_perl/5.005/i386-freebsd/Razor2/Client/Agent.pm > line 756. > Try
installing the latest Perl (at least 5.6.1) port on Freebsd and make sure you set the system to use
perl from ports (i.e. in the ports/lang/perl5/files directory run ./use.perl port. Reinstall the
relevant perl modules needed by razor and try again. Sven
------------------------------------------------------- This sf.net email is sponsored by:ThinkGeek
Welcome to geek heaven. http://thinkgeek.com/sf _______________________________________________
Razor-users mailing list Razor-users@lists.sourceforge.net
https://lists.sourceforge.net/lists/listinfo/r


<div class="alert alert-info">
<h2>Feature engineering </h2>

Below we use three different ways of extracting information from the data. These are three different representations of the dat from which the models can potentially learn. 
1.  **Numerical features** - we only look at the frequency of the "spammish signature". We don't use any information about the words in the emails.
2.  **Word counts** - We look at the counts of different words in each sample text. We do this once by recording whether a word "appears in email", and then using a relative frequency by comparing the frequnecy of a word in an email against all other emails.
3.  We use an existing LLM to transform the email text into a new more efficient and "informed" representation, called an **embedding**.

</div>


## Part 1: Extracting numeric features

We start with the ideas from the 2nd observation and create new features that count the abundance of different noise components in the text.

In [6]:
num_features_df = extract_numeric_features(df=df_source, with_labels=True)

Number of samples and columns of input: (5832, 2)
Number of columns: 2
Columns names:
spam_label, text

Numeric features extracted
Data size: (5832, 14)
Number of columns: 14
Columns names:
email_counts, html tag_counts, url_counts, Twitter username_counts, hashtag_counts
character_counts, word_counts, unique word_counts, punctuation mark_counts, uppercase word_counts
lowercase word_counts, digit_counts, alphabetic char_counts, spam_label
Numeric features saved to data/num_features.csv


## Part 2: Extracting features from text

Computers don't understand natural language and its unstructured form. So, how do we represent text?

### Bag of words

One of the simplest but in the early days of NLP effective and commonly used models to represent text for machine learning is the ***Bag of Words*** model ([link](https://en.wikipedia.org/wiki/Bag-of-words_model)). When using this model, we discard most of the structure of the input text (word order, chapters, paragraphs, sentences or formatting) and only count how often each word appears in each text. Discarding the structure and counting only word occurrences leads to the mental image of representing text as a "bag". 


**Example:** Let our toy corpus contain four documents.

```python
corpus = [
    'I enjoy paragliding.',
    'I do like NLP.',
    'I like deep learning.',
    'O Captain! my Captain!'
]
```

In [7]:
show_bag_of_words_vector()

,captain,deep,do,enjoy,i,learning,like,my,nlp,o,paragliding
Text,,,,,,,,,,,
I enjoy paragliding.,0,0,0,1,1,0,0,0,0,0,1
I do like NLP.,0,0,1,0,1,0,1,0,1,0,0
I like deep learning.,0,1,0,0,1,1,1,0,0,0,0
O Captain! my Captain!,2,0,0,0,0,0,0,1,0,1,0


In the table above, each column represents a word from the corpus and each row one of the four documents. The value in each cell represents the number of times that word appears in a specific document. For example, the fourth document has the word `captain` occurring twice and the words `my` and `O` occurring once.

The technical implementation of  Bag of Words is called a CountVectorizer. It converts each document into a rows of numbers, i.e. a numeric vector. Thus the name vectorizer.  

While this kind of transformation allows machine learning algorithms to process text data effectively, it has a drawback. It treats all words as independent and ignores the context in which they appear. For example, losing information about the order of the words in the text can change the meaning of a sentence. The sentences "I do like NLP", "Do I like NLP" or "NLP like I do" have the same set of words but different meanings. 

### TF-IDF

The **Term Frequency–Inverse Document Frequency** approach aims to address this limitation, by measuring how important a word is for a document relative to a collection of documents (the corpus). 

We use the implementation by scikit-learn. It calculates the TF-IDF score as the product of two quantities:

1. The **term frequency TF**, which is the ratio of: 
- the frequency of the word $w$ in the given document $d$ 
- the total number of words in the given document ($N(d)$), i.e. 
$$
TF(w, d) = \frac{f(w, d)}{N(d)}
$$

2. The second quantity is the (smoothed) ) **inverse document frequency IDF**, which is given by: 
$$
IDF(w, D) = \log\left(\frac{size(D)+1}{df(w, D)+1}\right)+1
$$ 
where $df(w, D)$ is the number of documents in the corpus $D$ that contain the word $w$. Adding `1` in the numerator and denominator keeps the IDF value finite and stable.

As mentioned, the TF-IDF score is simply their product: 

$$
TF-IDF(w,d) = TF(w, d) \times IDF(w, D)
$$

This way, **common words** that appear in **many documents** (small IDF) are **given less weight** while **rare words** that appear in only a **few documents** get a **higher weight** (high IDF).

**Example:** Let's compute the tf-idf scores of the words in our toy corpus. Remember that it contains four documents, hence $size(D)=4$. 

In [8]:
show_tfidf_vector()

,captain,deep,do,enjoy,i,learning,like,my,nlp,o,paragliding
Text,,,,,,,,,,,
I enjoy paragliding.,0.000000,0.00000,0.00000,0.644503,0.411378,0.00000,0.000000,0.000000,0.00000,0.000000,0.644503
I do like NLP.,0.000000,0.00000,0.57458,0.000000,0.366747,0.00000,0.453005,0.000000,0.57458,0.000000,0.000000
I like deep learning.,0.000000,0.57458,0.00000,0.000000,0.366747,0.57458,0.453005,0.000000,0.00000,0.000000,0.000000
O Captain! my Captain!,0.816497,0.00000,0.00000,0.000000,0.000000,0.00000,0.000000,0.408248,0.00000,0.408248,0.000000


For illustration, let's focus on the word "enjoy". It is listed once in the first document which contains 3 words, making $f("enjoy", 1) = 1$  $N(1) = 3$ and $TF("enjoy", 1) = 1/3$. On the other hand, we see that this word appears only in the first document of the corpus which contains four documents, therefore $df("enjoy", D) = 1$. If we were to compute this manually, the quantity is given by: 

In [4]:
import numpy as np

tf_enjoy_1  = 1/3
idf_enjoy_1 = (np.log((4+1)/(1+1))+1)

tf_idf_enjoy_1 = tf_enjoy_1 * idf_enjoy_1
tf_idf_enjoy_1

np.float64(0.638763577291385)

which is very close to the result from the function. The small divergence we see could be due to optimized implementations in the implemented versions or other issues. 

Below you can extract the text features using either the CountVectorizer (`vectorizer="count"`) or the TfidfVectorizer (`vectorizer="tfidf"`). Please note that this process can take a while depending on the memory of your machine, so be patient. For that reason, we have already pre-computed the features using `"tfidf"`and stored them in the `features` folder. You can load them using the command `load_feature_space(features="text")`.

In [9]:
%run data_prep_tools.py

text_features_df = extract_text_features(
    df_cleaned, vectorizer="tfidf", with_labels=True, store=True
)
text_features_df.shape

TF-IDF Vectorizer selected
Corpus data type:  <class 'scipy.sparse._csr.csr_matrix'>
0.8745627968449597 % are null values
Number of columns: 8219
First 5 names:
aall, aaron, abacha, abandon, abandoned
Last 5 columns:
zoom, zope, zurich, zyban, spam_label
Text features saved to  data/sparse_text_features_tfidf.npz


(5832, 8219)

## Part 3: Use LLMs to represent the text as a set of numbers

### Embeddings

The Bag of Words and TF-IDF approaches cannot capture the meaning of words or the relationships between them. They also lead to very high-dimensional and sparse representations of the text which are not very efficient and can lead to overfitting.
To address these limitations, we can use **embeddings** or transformer based models. Embeddings are denser vector representations of words are learned from large corpora of text. By representing similar words as similar vectors they can capture meaning and relationships in a continuous lower-dimensional vector space.

We have passed the email texts through a language model to generate the associated embeddings. Since the feature extraction takes some time we have stored these embeddings and made them available for you in the file named `email_embeddings.csv`.

You can load them using the command `load_feature_space(features="embeddings")`.

In [10]:
embeddings_df = load_feature_space(features="embedding")

Email embeddings loaded
Data includes labels in the column 'spam_label'
The data set has 5832 rows, 769 columns
